### Построение базовой модели

In [78]:
import numpy as np
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

seed = 42
np.random.seed = seed

In [2]:
with open('./data/train_data.json', 'r', encoding='utf-8') as file:
    train_data = json.load(file)

X_train = train_data['X']
y_train = train_data['y']

print(len(X_train), len(y_train))

with open('./data/test_data.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

X_test = test_data['X']
y_test = test_data['y']

print(len(X_test), len(y_test))

25000 25000
25000 25000


In [4]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',    # нормализация символов
    ngram_range=(1, 2),         # униграммы + биграммы
    min_df=2,                   # отсечение очень редкие токены
    max_df=0.95,                # отсекчение слишком частые токены
    max_features=50000,         # ограничение размерности
    sublinear_tf=True
)

pipe = Pipeline([
    ('tfidf', tfidf),
    ('clf', LinearSVC())
])

param_grid = [

    # LinearSVC
    {
        'clf': [LinearSVC()],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    },

    # Logistic Regression
    {
        'clf': [LogisticRegression(max_iter=2000, solver='liblinear')],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    }
]

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print('\nЛучшие параметры:')
print(grid.best_params_)

print('\nЛучший CV score:')
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print('\nTest Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

Fitting 3 folds for each of 384 candidates, totalling 1152 fits

Лучшие параметры:
{'clf': LinearSVC(), 'clf__C': 0.1, 'tfidf__max_df': 0.95, 'tfidf__max_features': 30000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}

Лучший CV score:
0.8822000396214552

Test Accuracy: 0.90188

Classification Report:
              precision    recall  f1-score   support

           0     0.9066    0.8961    0.9013     12500
           1     0.8973    0.9077    0.9024     12500

    accuracy                         0.9019     25000
   macro avg     0.9019    0.9019    0.9019     25000
weighted avg     0.9019    0.9019    0.9019     25000



#### После проведения тестов, лучшей базовой моделью оказалась LinearSVC с параметрами: 

- C = 0.1

#### и TfIdf векторизатором с параметрами:

- ngram_range = (1, 2)

- max_df = 0.9

- min_df = 1

- max_features = 30000

### Построение CNN модели

In [77]:
import re
import os
import random
from datetime import datetime

import itertools
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

Функции кодирования слов и создания словаря

In [42]:
_token_re = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    # простая англ. токенизация: слова и апострофы
    return _token_re.findall(text.lower())

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

def build_vocab(texts, min_freq=2, max_size=50000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    # самые частые слова в словарь, но частота их должна быть больше min_freq
    most_common = [w for w, c in counter.most_common() if c >= min_freq]
    # обрезаем словарь частых слов по максимальной длине словаря max_size
    if max_size is not None:
        most_common = most_common[:max_size]
    
    # создаем словарь индексов
    stoi = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    # присваиваем каждому слову уникальный номер
    for w in most_common:
        if w not in stoi:
            stoi[w] = len(stoi)
    return stoi

def encode(text, stoi, max_len):
    tokens = tokenize(text)
    # перевод слов в числа с помощью словаря stoi
    ids = [stoi.get(w, stoi[UNK_TOKEN]) for w in tokens]
    # обрезаем если отзыв слишком длинный
    ids = ids[:max_len]
    
    # если отзыв короткий, добавляем нули
    if len(ids) < max_len:
        ids += [stoi[PAD_TOKEN]] * (max_len - len(ids))
    return ids

Класс датасета и класс CNN модели

In [43]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        X = torch.tensor(encode(self.texts[index], self.stoi, self.max_len), dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        return X, y
    
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        num_classes: int = 2,
        kernel_sizes=(3, 4, 5),
        num_filters: int = 128,
        dropout: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=ks)
            for ks in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc =nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)

        pooled = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            p = torch.max(h, dim=2).values
            pooled.append(p)

        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        logits = self.fc(out)

        return logits

Функции обучения и валидации модели

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_sum, total = 0.0, 0
    all_preds, all_targets = [], []
    crossentropy = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = crossentropy(logits, y)

        loss_sum += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    avg_loss = loss_sum / total

    acc = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds)
    cl_report = classification_report(all_targets, all_preds, digits=4)

    return avg_loss, cl_report

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_sum, total = 0.0, 0
    all_preds, all_targets = [], []
    crossentropy = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = crossentropy(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    avg_loss = loss_sum / total

    acc = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds)
    cl_report = classification_report(all_targets, all_preds, digits=4)

    return avg_loss, cl_report

Создание словаря и модели, подбор гиперпараметров, обучение и оценка

In [45]:
grid = {
    'max_len': [200, 300, 400],
    'min_freq': [1, 2],
    'max_size': [50000],
    'batch_size': [16, 32],
    'embed_dim': [100, 128],
    'num_filters': [100, 128],
    'kernel_sizes': [(3, 4, 5), (2, 3, 4, 5)],
    'dropout': [0.3, 0.5],
    'lr': [1e-3, 5e-4],
    'epochs': [6],
}

os.makedirs('logs', exist_ok=True)

keys = list(grid.keys())
combos = list(itertools.product(*[grid[k] for k in keys]))

best_test_loss = 100

for i, values in enumerate(combos, start=1):

    config = dict(zip(keys, values))

    stoi = build_vocab(X_train, config['min_freq'], config['max_size'])
    vocab_size = len(stoi)

    train_ds = TextDataset(X_train, y_train, stoi, config['max_len'])
    test_ds = TextDataset(X_test, y_test, stoi, config['max_len'])

    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    modelCNN = TextCNN(
        vocab_size=vocab_size,
        embed_dim=config['embed_dim'],
        num_classes=2,
        kernel_sizes=config['kernel_sizes'],
        num_filters=config['num_filters'],
        dropout=config['dropout'],
        pad_idx=stoi[PAD_TOKEN],
    ).to(device)

    optimizer_AdamW = torch.optim.AdamW(modelCNN.parameters(), lr=config['lr'])

    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    run_name = f'textcnn_{timestamp}_bs{config['batch_size']}_mf{config['min_freq']}_ms{config['max_size']}_lr{config['lr']} \
    _len{config['max_len']}_ed{config['embed_dim']}_nf{config['num_filters']}_do{config['dropout']}'
    log_path = os.path.join('logs', run_name + '.txt')
    ckpt_path = os.path.join('logs', run_name + '.pt')

    with open(log_path, 'w', encoding='utf-8') as file:
        file.write('TextCNN Training Log\n\n')

        file.write('PARAMETRS:\n')
        file.write(f'max_len - {config['max_len']}\n')
        file.write(f'min_freq - {config['min_freq']}\n\n')
        file.write(f'max_size - {config['max_size']}\n\n')
        file.write(f'batch_size - {config['batch_size']}\n')
        file.write(f'embed_dim - {config['embed_dim']}\n')
        file.write(f'num_filters - {config['num_filters']}\n')
        file.write(f'kernel_sizes - {config['kernel_sizes']}\n')
        file.write(f'dropout - {config['dropout']}\n')
        file.write(f'lr - {config['lr']}\n\n')

        file.write(f'Start time: {timestamp}\n\n')

    test_loss_prev = 0
    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_report = train_one_epoch(modelCNN, train_loader, optimizer_AdamW, device)
        test_loss, test_report = evaluate(modelCNN, test_loader, device)

        with open(log_path, 'a', encoding='utf-8') as file:
            file.write(f'EPOCH {epoch:02d} TRAIN\n')
            file.write(f'train loss {train_loss:.4f}\n')
            file.write(train_report)
            file.write('                         ------\n')
            file.write(f'EPOCH {epoch:02d} TEST\n')
            file.write(f'test loss {test_loss:.4f}\n')
            file.write(test_report)
            file.write('                         ------\n\n')

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(modelCNN.state_dict(), ckpt_path)

        if test_loss_prev < test_loss and epoch > 1:
            with open(log_path, 'a', encoding='utf-8') as file:
                file.write('Качество модели стало ухудшаться')
            break
        else:
            test_loss_prev = test_loss

KeyboardInterrupt: 

Моделью с самым низким loss на тестовых данных оказалась модель с параметрами:
- max_len - 400
- min_freq - 2
- max_size - 50000
- batch_size - 32
- embed_dim - 128
- num_filters - 100
- kernel_sizes - (2, 3, 4, 5)
- dropout - 0.3
- lr - 0.0005

EPOCH 05 TEST

test loss 0.2857

              precision    recall  f1-score   support
           0     0.8938    0.8641    0.8787     12500
           1     0.8684    0.8973    0.8826     12500

    accuracy                         0.8807     25000
    macro avg     0.8811    0.8807    0.8806     25000
    weighted avg     0.8811    0.8807    0.8806     25000

Качество модели TextCNN оказалосьхуже, чем у baseline

### Построение LSTM модели

In [92]:
class RNNClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_size: int = 128,
        num_layers: int = 1,
        rnn_type: str = 'lstm',
        bidirectional: bool = True,
        dropout: float = 0.5,
        pad_idx: int = 0,
        num_classes: int = 2
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, pad_idx)

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if rnn_type.lower() == 'gru':
            self.rnn = nn.GRU(
                input_size=embed_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=rnn_dropout
            )
        else:
            self.rnn = nn.LSTM(
                input_size=embed_dim,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=rnn_dropout
            )

        self.bidirectional = bidirectional
        self.rnn_type = rnn_type.lower()

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, h = self.rnn(emb)

        if self.rnn_type == 'lstm':
            h_n = h[0]
        else:
            h_n = h

        if self.bidirectional:
            h_last_fwd = h_n[-2]
            h_last_bwd = h_n[-1]
            feat = torch.cat([h_last_fwd, h_last_bwd], dim = 1)
        else:
            feat = h_n[-1]

        feat = self.dropout(feat)
        logits = self.fc(feat)

        return logits

def train_model(X_train, y_train, X_test, y_test, config, output_dir, best_test_loss = None):

    stoi = build_vocab(X_train, config['min_freq'], config['max_size'])

    train_ds = TextDataset(X_train, y_train, stoi, config['max_len'])
    test_ds = TextDataset(X_test, y_test, stoi, config['max_len'])

    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    modelRNN = RNNClassifier(
        vocab_size=len(stoi),
        embed_dim=config['embed_dim'],
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        rnn_type=config['rnn_type'],
        bidirectional=config['bidirectional'],
        dropout=config['dropout'],
        pad_idx=stoi[PAD_TOKEN],
        num_classes=2
    ).to(device)

    optimizer_AdamW = torch.optim.AdamW(modelRNN.parameters(), lr=config['lr'])

    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    run_name = f'{config['rnn_type']}_{timestamp}_bs{config['batch_size']}_mf{config['min_freq']}_ms{config['max_size']}_lr{config['lr']} \
    _len{config['max_len']}_ed{config['embed_dim']}_nf{config['hidden_size']}_do{config['dropout']}'
    log_path = os.path.join(output_dir, run_name + '.txt')
    ckpt_path = os.path.join(output_dir, run_name + '.pt')

    with open(log_path, 'w', encoding='utf-8') as file:
        file.write(f'{config['rnn_type']} Training Log\n\n')

        file.write('PARAMETRS:\n')
        file.write(f'bidirectional - {config['bidirectional']}\n')
        file.write(f'max_len - {config['max_len']}\n')
        file.write(f'min_freq - {config['min_freq']}\n')
        file.write(f'max_size - {config['max_size']}\n')
        file.write(f'batch_size - {config['batch_size']}\n')
        file.write(f'embed_dim - {config['embed_dim']}\n')
        file.write(f'hidden_size - {config['hidden_size']}\n')
        file.write(f'num_layers - {config['num_layers']}\n')
        file.write(f'dropout - {config['dropout']}\n')
        file.write(f'lr - {config['lr']}\n\n')

        file.write(f'Start time: {timestamp}\n\n')

    test_loss_prev = 0
    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_report = train_one_epoch(modelRNN, train_loader, optimizer_AdamW, device)
        test_loss, test_report = evaluate(modelRNN, test_loader, device)

        with open(log_path, 'a', encoding='utf-8') as file:
            file.write(f'EPOCH {epoch:02d} TRAIN\n')
            file.write(f'train loss {train_loss:.4f}\n')
            file.write(train_report)
            file.write('                         ------\n')
            file.write(f'EPOCH {epoch:02d} TEST\n')
            file.write(f'test loss {test_loss:.4f}\n')
            file.write(test_report)
            file.write('                         ------\n\n')

        if best_test_loss != None and test_loss < best_test_loss[0]:
            best_test_loss[:] = [test_loss]
            torch.save(modelCNN.state_dict(), ckpt_path)

        if test_loss_prev < test_loss - 0.02 and epoch > 1:
            with open(log_path, 'a', encoding='utf-8') as file:
                file.write('Качество модели стало ухудшаться')
            break
        else:
            test_loss_prev = test_loss

In [ ]:
grid = {
    'max_len': [400],
    'min_freq': [2],
    'max_size': [50000],
    'batch_size': [32],
    'embed_dim': [100, 128],
    'hidden_size': [100, 128],
    'num_layers': [1],
    'dropout': [0.3, 0.5],
    'lr': [1e-3, 5e-4],
    'epochs': [8],
    'rnn_type': ['lstm', 'gru'],
    'bidirectional': [True, False],
}

os.makedirs('rnn_logs', exist_ok=True)

keys = list(grid.keys())
combos = list(itertools.product(*[grid[k] for k in keys]))

best_test_loss = [100]
for i, values in enumerate(combos, start=1):

    config = dict(zip(keys, values))
    train_model(X_train, y_train, X_test, y_test, config, 'rnn_logs', best_test_loss)

print(best_test_loss)

[0.33612788050174713]


Лучшей оказалась модель Bidirectional GRU с параметрами:
- max_len - 400
- min_freq - 2
- max_size - 50000
- batch_size - 32
- embed_dim - 100
- hidden_size - 100
- num_layers - 1
- dropout - 0.3
- lr - 0.001

Результат на тесте:

EPOCH 04 TEST

test loss 0.3546

              precision    recall  f1-score   support
           0     0.8555    0.8767    0.8660     12500
           1     0.8736    0.8519    0.8626     12500

    accuracy                         0.8643     25000
    macro avg     0.8645    0.8643    0.8643     25000
    weighted avg     0.8645    0.8643    0.8643     25000

Качество реккурентных моделей оказалось хуже, чем у baseline

### Построение BERT подобной модели